# 09 — Yıllar Arası CV (Hafta 11)

**Hedef:** RF modelin **temporal tutarlılığı**. Hafta 10'da 5-yıl medyan LST üzerinde eğitilen model, **her yıl için** ayrı LST gerçekleriyle nasıl uyuşur?

**Mantık:** Diğer feature'lar (NDVI, Albedo, Impervious, building, road, DTC) yıllık değişmez (5-yıl medyan/sabit), bu yüzden modelin çıktısı sabit. Yıl-by-yıl LST'yi hedef alarak modelin temporal generalization'ını ölçeriz.

**Persistence analizi:** UTPM'in temel hipotezi — bazı hücreler **tutarlı şekilde** sıcaktır (kentsel ısı kalıcılığı). Bunu ölçmek için:
- Her yıl için LST quartile'ları (1=en serin, 4=en sıcak)
- 5/5 yıl Q4'te = persistent hot spot
- 5/5 yıl Q1'de = persistent cold spot

**Önkoşul:** `grid_30m_modeling.gpkg` (Hafta 9), `results/rf_model.pkl` (Hafta 10), GEE auth.

**Çıktılar:**
- `data/processed/lst_summer_2020.tif` ... `lst_summer_2024.tif` (5 ayrı yıllık kompozit)
- `data/processed/grid_30m_yearly_lst.gpkg` — grid + 5 yıllık LST kolon
- `data/processed/grid_30m_persistence.gpkg` — persistence skorlu grid
- `results/cross_year_validation.json`
- `tables/09_cross_year_metrics.csv`
- `figures/09_*.png`

In [ ]:
import sys, os, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.config import (
    DATA_GRID, DATA_PROCESSED, RESULTS, FIGURES, TABLES,
    GRID_30M_MODELING, MODEL_PKL, GEE_PROJECT,
    LST_YEARS, LST_MONTHS, CLOUD_COVER_THRESHOLD,
    LST_YEARLY_RASTERS, GRID_30M_YEARLY,
    CROSS_YEAR_VALIDATION_JSON, PERSISTENCE_GPKG,
    SELECTED_FEATURES, TARGET_COLUMN, CRS_PROJECTED,
)
from src.gee_utils import init_ee, boundary_to_ee_geometry, summer_median_lst
from src.variables import zonal_stats_to_grid
from src.modeling import prepare_modeling_matrix

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# --- Yıllık LST kompozitlerini indir ---
import ee
project_id = GEE_PROJECT or os.environ.get("GEE_PROJECT")
init_ee(project_id)

boundary = gpd.read_file(DATA_GRID / "pilot_boundary.gpkg")
region_ee = boundary_to_ee_geometry(boundary)

import geemap
for year in LST_YEARS:
    out_path = LST_YEARLY_RASTERS[year]
    if out_path.exists():
        print(f"{year}: cache hit")
        continue
    print(f"{year}: indiriliyor...")
    img = summer_median_lst(
        region=region_ee, years=[year], months=LST_MONTHS,
        cloud_cover_max=CLOUD_COVER_THRESHOLD,
    )
    geemap.ee_export_image(
        ee_object=img, filename=str(out_path),
        scale=30, region=region_ee, file_per_band=False,
        crs="EPSG:32636",
    )
    print(f"  saved")

In [ ]:
# --- Zonal stats: her yıl için 30m grid'e LST ekle ---
grid = gpd.read_file(GRID_30M_MODELING)
print(f"Grid: {len(grid):,} hücre")

for year in LST_YEARS:
    grid = zonal_stats_to_grid(
        grid=grid, raster_path=LST_YEARLY_RASTERS[year],
        stats=("mean",), prefix=f"lst_{year}_",
    )

year_cols = [f"lst_{y}_mean" for y in LST_YEARS]
print("\nYıl bazlı LST özeti:")
print(grid[year_cols].describe().round(2).to_string())

In [ ]:
# --- Mahalle + RF tahmin ---
imar = gpd.read_file(DATA_PROCESSED / "imar_pilot.gpkg")
joined = gpd.sjoin_nearest(grid[["cell_id", "geometry"]],
                            imar[["mahalle", "geometry"]],
                            how="left", max_distance=200)
grid["mahalle"] = grid["cell_id"].map(joined.groupby("cell_id")["mahalle"].first())

model = joblib.load(MODEL_PKL)
X, y, feat_names = prepare_modeling_matrix(
    grid, features=SELECTED_FEATURES, target=TARGET_COLUMN,
    fillna_zero=True, add_saturated_flag=True,
)
grid["predicted_lst"] = model.predict(X)
print(f"RF tahmin: {grid['predicted_lst'].describe().round(2).to_dict()}")

In [ ]:
# --- Yıl-by-yıl validation ---
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

metrics = []
for year in LST_YEARS:
    actual = grid[f"lst_{year}_mean"]
    pred = grid["predicted_lst"]
    mask = actual.notna()
    a, p = actual[mask].to_numpy(), pred[mask].to_numpy()
    metrics.append({
        "yil": year,
        "n_cells": int(mask.sum()),
        "actual_mean": round(float(a.mean()), 2),
        "actual_std": round(float(a.std()), 2),
        "RMSE": round(float(np.sqrt(mean_squared_error(a, p))), 3),
        "MAE": round(float(mean_absolute_error(a, p)), 3),
        "R2": round(float(r2_score(a, p)), 3),
    })

ym = pd.DataFrame(metrics)
print("Yıllar arası validation:")
print(ym.to_string(index=False))
ym.to_csv(TABLES / "09_cross_year_metrics.csv", index=False, encoding="utf-8-sig")

In [ ]:
# --- Yıllar arası LST korelasyonu (persistence göstergesi) ---
yearly_corr = grid[year_cols].corr().round(3)
print("Yıllar arası LST korelasyonu:")
print(yearly_corr.to_string())

mask_off = np.ones_like(yearly_corr.values, dtype=bool)
np.fill_diagonal(mask_off, False)
off = yearly_corr.values[mask_off]
print(f"\nDiagonal-dışı: min={off.min():.3f}, mean={off.mean():.3f}, max={off.max():.3f}")
print("Yorum: yüksek diagonal-dışı = mekânsal LST deseni yıllar arası kalıcı")

In [ ]:
# --- Persistence skorları ---
yearly_array = grid[year_cols].to_numpy()
grid["lst_yearly_mean"]  = np.nanmean(yearly_array, axis=1)
grid["lst_yearly_std"]   = np.nanstd(yearly_array, axis=1)
grid["lst_yearly_min"]   = np.nanmin(yearly_array, axis=1)
grid["lst_yearly_max"]   = np.nanmax(yearly_array, axis=1)
grid["lst_yearly_range"] = grid["lst_yearly_max"] - grid["lst_yearly_min"]

qs = pd.DataFrame()
for year in LST_YEARS:
    qs[year] = pd.qcut(grid[f"lst_{year}_mean"], 4, labels=[1, 2, 3, 4])
grid["years_in_top_quartile"] = (qs == 4).sum(axis=1)
grid["years_in_bottom_quartile"] = (qs == 1).sum(axis=1)

print("Persistence dağılımı:")
print(f"  Tüm 5 yıl Q4'te (persistent HOT):  {(grid['years_in_top_quartile']==5).sum():,}")
print(f"  4/5 yıl Q4'te:                      {(grid['years_in_top_quartile']==4).sum():,}")
print(f"  Tüm 5 yıl Q1'de (persistent COLD): {(grid['years_in_bottom_quartile']==5).sum():,}")
print(f"  4/5 yıl Q1'de:                      {(grid['years_in_bottom_quartile']==4).sum():,}")
print(f"  Yıllık std medyan:                  {grid['lst_yearly_std'].median():.3f}°C")

In [ ]:
# --- Görselleştirme: yıllar arası tutarlılık + persistence haritası ---
fig, axes = plt.subplots(2, 2, figsize=(15, 13))

# 1. Yıl-by-yıl R²
ax = axes[0, 0]
ax.bar(ym["yil"].astype(str), ym["R2"], color="#2A9D8F", edgecolor="black")
ax.set_xlabel("Yıl")
ax.set_ylabel("R² (RF tahmin vs gerçek yıllık LST)")
ax.set_title("Yıllar arası model tutarlılığı")
for i, v in enumerate(ym["R2"]):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
ax.set_ylim(0, 1)

# 2. Yıllar arası korelasyon
ax = axes[0, 1]
sns.heatmap(yearly_corr, annot=True, fmt=".3f", cmap="YlOrRd",
            ax=ax, cbar_kws={"label": "Pearson r"}, vmin=0.5, vmax=1.0,
            annot_kws={"size": 9})
ax.set_title("Yıllar arası LST korelasyonu")

# 3. Persistence — top quartile sayısı
ax = axes[1, 0]
grid.plot(column="years_in_top_quartile", cmap="OrRd", ax=ax, linewidth=0,
          legend=True, legend_kwds={"label": "5 yılda Q4'te kaç yıl", "shrink": 0.6},
          vmin=0, vmax=5)
boundary.to_crs(CRS_PROJECTED).boundary.plot(ax=ax, color="#0a9396", linewidth=0.8)
ax.set_title("Persistent HOT spots (en sıcak %25)")
ax.set_aspect("equal")
ax.ticklabel_format(style="plain")

# 4. Yıllık LST std (variability)
ax = axes[1, 1]
vals = grid["lst_yearly_std"].dropna()
grid.plot(column="lst_yearly_std", cmap="viridis", ax=ax, linewidth=0,
          legend=True, legend_kwds={"label": "5 yıl std (°C)", "shrink": 0.6},
          vmin=np.percentile(vals, 2), vmax=np.percentile(vals, 98))
boundary.to_crs(CRS_PROJECTED).boundary.plot(ax=ax, color="#0a9396", linewidth=0.8)
ax.set_title("Yıllar arası LST varyabilitesi")
ax.set_aspect("equal")
ax.ticklabel_format(style="plain")

plt.tight_layout()
plt.savefig(FIGURES / "09_cross_year_overview.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# --- Kaydet ---
out_cols = ([c for c in grid.columns if c.startswith("lst_") or c.startswith("years_")]
            + ["cell_id", "mahalle", "predicted_lst", "geometry"])
out_cols = list(dict.fromkeys(out_cols))  # uniqueness koru
grid[out_cols].to_file(GRID_30M_YEARLY, driver="GPKG")
print(f"Kaydedildi: {GRID_30M_YEARLY.name} ({GRID_30M_YEARLY.stat().st_size/1024/1024:.2f} MB)")

persistence_cols = ["cell_id", "mahalle", "geometry",
                    "lst_yearly_mean", "lst_yearly_std", "lst_yearly_range",
                    "years_in_top_quartile", "years_in_bottom_quartile"]
grid[persistence_cols].to_file(PERSISTENCE_GPKG, driver="GPKG")
print(f"Kaydedildi: {PERSISTENCE_GPKG.name}")

val_data = {
    "yearly_metrics": metrics,
    "yearly_correlation": yearly_corr.to_dict(),
    "off_diagonal_corr_mean": float(off.mean()),
    "persistent_hot_count": int((grid['years_in_top_quartile']==5).sum()),
    "persistent_cold_count": int((grid['years_in_bottom_quartile']==5).sum()),
    "yearly_std_median": float(grid['lst_yearly_std'].median()),
}
with open(CROSS_YEAR_VALIDATION_JSON, "w", encoding="utf-8") as f:
    json.dump(val_data, f, indent=2, ensure_ascii=False)
print(f"Kaydedildi: {CROSS_YEAR_VALIDATION_JSON.name}")

## Özet

- 5 yıl × LST kompoziti GEE'den indirildi (30 m, yaz medyan, bulut < %10).
- Mevcut RF model her yıl için tahmin yaptı; yıllık RMSE/MAE/R² metrikleri hesaplandı.
- Yıllar arası LST korelasyon matrisi → mekânsal desenin temporal kalıcılığı.
- Persistence skorları: kaç yıl Q4 (en sıcak) veya Q1 (en serin) quartile'larında.

## UTPM ile bağlantı

Hafta 12'de UTPM (Urban Thermal Persistence Model) lineer indeksi:  
`UTPM = w_albedo·albedo + w_imp·imp + w_dtc·dtc + ...` (RF permutation importance ağırlıklarıyla)

Bu Hafta 11 sonuçları **persistence skorlarını doğrular** — eğer modelin tahmini ve gerçek yıllık LST yıllar arası tutarlıysa, lineer UTPM indeksi de aynı persistent hot/cold spotları ortaya çıkarmalı.

## Sonraki Adım

`10_shap_utpm.ipynb` (Hafta 12) — SHAP değerleri + UTPM lineer indeks + Moran's I + Jenks sınıflama.